In [ ]:
import pandas as pd
import altair as alt
import sys
import os

In [4]:
# load data

sys.path.append(os.path.abspath(os.path.join('..', 'src')))
from load_and_merge import load_and_merge
all = load_and_merge()
all

,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,...,Decompression,Best Lap Time,Number of Laps,Moving Time,Elapsed Time,Min Elevation,Max Elevation,Total Strokes,Min Temp,Max Temp
0,Running,2025-04-26 13:19:07,False,Chicago Running,8.24,"1,000",01:06:06,166,178,3.8,...,No,00:02:03.9,9,01:05:59,01:06:06,571,645,NaN,NaN,NaN
1,Running,2025-04-21 11:15:07,False,Hopkinton Running,26.49,"3,209",04:19:31,164,184,5.0,...,No,00:04:51.3,27,04:19:25,04:19:31,16,453,NaN,NaN,NaN
2,Running,2025-04-19 08:15:36,False,Providence Running,5.11,658,00:43:30,160,172,3.4,...,No,00:00:57.0,6,00:43:26,00:43:30,-3,52,NaN,NaN,NaN
3,Running,2025-04-17 07:08:05,False,Chicago Running,6.25,721,00:45:02,170,187,3.9,...,No,00:01:54.5,7,00:44:59,00:45:02,577,648,NaN,NaN,NaN
4,Running,2025-04-15 07:19:23,False,Chicago Running,4.15,499,00:31:58,165,182,3.4,...,No,00:01:03.8,5,00:31:57,00:31:58,579,653,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448,Running,2022-08-29 07:55:36,False,Islip Running,7.47,969,01:08:04,161,185,4.5,...,No,00:04:05.3,8,--:--:--,01:08:04,-1,19,--,86.0,91.4
449,Running,2022-08-28 07:43:58,False,Islip Running,7.24,883,01:09:57,143,164,3.4,...,No,00:02:02.6,8,--:--:--,01:09:57,154,180,--,80.6,86.0
450,Running,2022-08-27 06:44:39,False,New York Running,6.23,772,00:50:05,167,188,4.8,...,No,00:01:47.6,7,--:--:--,00:50:05,12,103,--,84.2,86.0
451,Hiking,2022-08-23 00:28:29,False,Rombo Hiking,3.03,"2,210",06:56:40,108,135,2.2,...,No,00:00:03.4,4,00:56:14,06:56:42,"15,985","19,251",--,59.0,80.6


In [5]:
import pandas as pd
import altair as alt

df = all

def time_to_minutes(pace):
    """Convert Avg Pace (in M:SS) to a numeric value (total minutes)"""
    parts = pace.split(':')
    if len(parts) == 2:
        minutes = float(parts[0])
        seconds = float(parts[1])
        return minutes + seconds/60
    return None

def time_to_hours(pace):
    """Convert Avg Pace (in HH:MM:SS) to a numeric value (total hours)"""
    parts = pace.split(':')
    if len(parts) == 3:
        hours = float(parts[0])
        minutes = float(parts[1])
        seconds = float(parts[2])
        return hours + minutes/60 + seconds/3600
    return None

df['avg_pace_minutes'] = df['Avg Pace'].apply(time_to_minutes)
df['time_hours'] = df['Time'].apply(time_to_hours)

df['Calories'] = pd.to_numeric(df['Calories'], errors='coerce')

df['cal/hr'] = df['Calories'] / df['time_hours']


brush = alt.selection_interval(
    encodings=['x','y'],
    resolve='intersect')

col = alt.condition(brush, alt.Color('Location:N'), alt.ColorValue('rgba(200, 200, 200, 0.4)'))

# Base histogram: shows all data in gray
p11_base = alt.Chart(df).mark_bar(color='gray').encode(
    x=alt.X('Distance:Q', bin=alt.Bin(maxbins=28), title='Distance'),
    y=alt.Y('count()', title='Count')
)

# Overlay: shows only the selected data, colored by location
p11_selected = alt.Chart(df).mark_bar().encode(
    x=alt.X('Distance:Q', bin=alt.Bin(maxbins=28)),
    y='count()',
    color=alt.Color('Location:N')
).transform_filter(brush)

# Combine the layers
p11 = (p11_base + p11_selected).add_params(brush).properties(width=430, height=260)

p21 = alt.Chart(df).mark_circle().encode(
    x=alt.X('Date:T', title='Date'),
    y=alt.Y('Distance:Q', title='Distance '),
    color=col
).add_params(brush).properties(width=550, height=260)

p31 = alt.Chart(df).mark_circle().encode(
    x='Distance:Q',
    y=alt.Y('avg_pace_minutes:Q', title='Average Pace (minutes)', scale=alt.Scale(domain=[7,11])),
    color=col
).add_params(brush).properties(width=350, height=260)

p41 = alt.Chart(df).mark_circle().encode(
    x='Date:T',
    y=alt.Y('avg_pace_minutes:Q', title='Average Pace (minutes)', scale=alt.Scale(domain=[7,11])),
    color=col
).add_params(brush).properties(width=350, height=260)

p51 = alt.Chart(df).mark_circle().encode(
    x='Distance:Q',
    y=alt.Y('Avg HR:Q',scale=alt.Scale(domain=[130,190]))
    ,color=col
).add_params(brush).properties(width=350, height=260)

fig = alt.vconcat(p11, p21, p31, p41, p51)

fig


/var/folders/dj/8cmdh9295h1f35fm67cwx_jm0000gn/T/ipykernel_64353/733292620.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Location'] = df['Title'].str.replace('Running', '').str.strip()
/var/folders/dj/8cmdh9295h1f35fm67cwx_jm0000gn/T/ipykernel_64353/733292620.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['Date'] = pd.to_datetime(df['Date'])
/var/folders/dj/8cmdh9295h1f35fm67cwx_jm0000gn/T/ipykernel_64353/733292620.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of 

alt.VConcatChart(...)

In [2]:
df

,Activity Type,Date,Favorite,Title,Distance,Calories,Time,Avg HR,Max HR,Aerobic TE,...,Number of Laps,Max Temp,Moving Time,Elapsed Time,Min Elevation,Max Elevation,Location,avg_pace_minutes,time_hours,cal/hr
0,Running,2025-03-15 10:46:42,False,Naples Running,5.52,693.0,00:43:24,177,192,3.9,...,6,--,00:43:22,00:43:24,1,24,Naples,7.866667,0.723333,958.064516
1,Running,2025-03-14 10:16:04,False,Naples Running,5.62,697.0,00:43:49,175,187,3.9,...,6,--,00:43:43,00:43:49,3,22,Naples,7.800000,0.730278,954.431343
2,Running,2025-03-11 11:29:36,False,Chicago Running,4.19,520.0,00:32:52,172,183,3.5,...,5,--,00:32:44,00:32:52,711,783,Chicago,7.850000,0.547778,949.290061
3,Running,2025-03-08 13:02:31,False,Chicago Running,15.13,NaN,02:01:16,178,191,5.0,...,16,--,02:01:13,02:01:16,577,635,Chicago,8.016667,2.021111,NaN
4,Running,2025-03-06 07:09:09,False,Chicago Running,6.01,737.0,00:47:17,166,187,3.8,...,7,--,00:46:44,00:47:23,570,649,Chicago,7.866667,0.788056,935.213253
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427,Running,2022-08-30 07:28:47,False,Islip Running,7.41,947.0,01:08:45,155,174,4.1,...,8,89.6,--:--:--,01:08:45,6,24,Islip,9.283333,1.145833,826.472727
428,Running,2022-08-29 07:55:36,False,Islip Running,7.47,969.0,01:08:04,161,185,4.5,...,8,91.4,--:--:--,01:08:04,-1,19,Islip,9.116667,1.134444,854.162586
429,Running,2022-08-28 07:43:58,False,Islip Running,7.24,883.0,01:09:57,143,164,3.4,...,8,86.0,--:--:--,01:09:57,154,180,Islip,9.666667,1.165833,757.398142
430,Running,2022-08-27 06:44:39,False,New York Running,6.23,772.0,00:50:05,167,188,4.8,...,7,86.0,--:--:--,00:50:05,12,103,New York,8.033333,0.834722,924.858569
